# Signal Generation and Walk-Forward Backtest

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signals.backtest import WalkForwardBacktest
from signals.portfolio import risk_parity_weights, apply_constraints

rng = np.random.default_rng(42)
T = 756
assets = ['SPX', 'HY_OAS', 'VIX', 'COPPER', 'DXY']
dates = pd.date_range('2018-01-01', periods=T, freq='B')

# Synthetic returns with a known lead (asset 0 leads asset 1)
base = rng.standard_normal((T, len(assets))) * 0.01
base[1:, 1] += 0.3 * base[:-1, 0]  # Asset 0 leads asset 1
returns = pd.DataFrame(base, index=dates, columns=assets)

print(f'Returns panel: {returns.shape}')

In [ ]:
def simple_signal(returns_slice: pd.DataFrame) -> dict:
    """Simple momentum signal: buy assets with positive trailing 21-day return."""
    lookback = min(21, len(returns_slice))
    trailing = returns_slice.iloc[-lookback:].sum()
    weights = (trailing > 0).astype(float)
    n_pos = weights.sum()
    if n_pos > 0:
        weights = weights / n_pos
    return weights.to_dict()

bt = WalkForwardBacktest(
    returns=returns,
    signal_func=simple_signal,
    initial_window=252,
    step_size=21,
)
bt.run()
metrics = bt.compute_metrics()
print('Performance Metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
eq = bt.equity_curve()
dd = bt.drawdown_series()

axes[0].plot(eq, label='Portfolio', color='blue')
axes[0].set_title('Equity Curve')
axes[0].legend()

axes[1].fill_between(dd.index, dd.values, 0, color='red', alpha=0.4)
axes[1].set_title('Drawdown')

plt.tight_layout()
plt.show()